# Medición reproducible del tiempo de respuesta — MLP V1 (PoC UNA)

**Versión corregida para Google Colab · modelo V1 reentrenado y persistido.**

Este notebook **NO vuelve a entrenar el modelo**. Abre el archivo `modelo_desercion_mlp_v1.keras` de la **segunda ejecución de V1**, carga el TEST ya procesado, comprueba que `y` no se utiliza como predictor y mide el tiempo de inferencia con TensorFlow. Incluye resultados, gráfico, registro del entorno, hashes SHA-256 y exportación de evidencia.

**Aclaración de trazabilidad:** el archivo `.keras` pertenece a una ejecución posterior de V1, distinta de la corrida original de comparación. El notebook calcula las métricas del modelo realmente cargado y no le atribuye las métricas de otra ejecución.

**Alcance:** latencia local de inferencia con datos preparados, no un sistema en producción; no incluye imputación, escalado ni codificación de datos crudos ni comunicación de red. La lectura del CSV ya procesado se mide por separado.

## 1. Archivos y entorno de Google Colab

1. Abre este `.ipynb` en **Google Colab**.
2. Selecciona **Entorno de ejecución → Cambiar tipo de entorno de ejecución → Python 3**. CPU basta para esta PoC; si eliges GPU, se registrará.
3. Ejecuta las celdas **en orden**. La siguiente celda abrirá el selector de archivos si faltan el `.keras` o el `.csv`. Selecciona ambos desde tu computadora. **No hay que escribir ni editar rutas.**
4. Guarda el notebook **después de ejecutarlo**, para conservar sus salidas, y descarga el ZIP de evidencia.

En Colab los archivos cargados en `/content/` son temporales: habrá que volver a subirlos en otra sesión.

In [ ]:
# Versión pública: se omite el montaje automático de Google Drive.


In [ ]:
from pathlib import Path
import hashlib, json, os, platform, sys, time, zipfile
from datetime import datetime

import numpy as np
import pandas as pd

MODEL_NAME = 'modelo_desercion_mlp_v1.keras'
TEST_NAME = 'REGEN_basev6_X_test_ready.csv'

# En Colab se usa /content. Fuera de Colab, usa el directorio de trabajo actual.
IN_COLAB = Path('/content').exists()
BASE_DIR = Path('/content') if IN_COLAB else Path.cwd()

faltantes = [nombre for nombre in (MODEL_NAME, TEST_NAME) if not (BASE_DIR / nombre).is_file()]
if faltantes and IN_COLAB:
    os.chdir(BASE_DIR)  # El selector de Colab siempre guarda los archivos aquí.
    print('Sube desde tu equipo los archivos requeridos: ' + ', '.join(faltantes))
    from google.colab import files
    files.upload()  # Guarda en /content sin que debas editar las rutas.

MODEL_PATH = BASE_DIR / MODEL_NAME
TEST_PATH = BASE_DIR / TEST_NAME
OUTPUT_DIR = BASE_DIR / 'evidencia_latencia_mlp_v1'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not MODEL_PATH.is_file() or not TEST_PATH.is_file():
    raise FileNotFoundError(
        'Faltan archivos en ' + str(BASE_DIR) + '. ' +
        'Nombres exactos requeridos: ' + MODEL_NAME + ' y ' + TEST_NAME + '. ' +
        'Si Colab agregó (1) al nombre, renombra el archivo desde la pestaña Archivos.'
    )
print('Modelo localizado:', MODEL_PATH)
print('TEST localizado:', TEST_PATH)

# Importar TensorFlow después de verificar los archivos; no se necesita reentrenar.
try:
    import tensorflow as tf
except ImportError as exc:
    raise ImportError('Este notebook requiere TensorFlow. En Google Colab estándar suele venir instalado. '
                      'Si no está disponible, ejecuta %pip install tensorflow y reinicia el entorno.') from exc

from sklearn.metrics import (confusion_matrix, accuracy_score, recall_score, precision_score,
                             f1_score, roc_auc_score, average_precision_score)
from IPython.display import display

print('TensorFlow:', tf.__version__)
print('GPU detectadas:', [str(g) for g in tf.config.list_physical_devices('GPU')])

## 2. Validación del modelo guardado y su TEST

La etiqueta `y` se separa del CSV **antes** de cualquier inferencia. La evaluación calculada aquí describe **únicamente el `.keras` subido**. Las métricas previas se indican solo como referencia de otra ejecución, no como requisito para medir latencia.

In [ ]:
def sha256(ruta, bloque=2**20):
    h = hashlib.sha256()
    with open(ruta, 'rb') as f:
        for pedazo in iter(lambda: f.read(bloque), b''):
            h.update(pedazo)
    return h.hexdigest()

inicio_carga = time.perf_counter_ns()
modelo = tf.keras.models.load_model(str(MODEL_PATH), compile=False)
tiempo_carga_ms = (time.perf_counter_ns() - inicio_carga)/1e6

raw = pd.read_csv(TEST_PATH, encoding='utf-8-sig')
if 'y' not in raw.columns:
    raise ValueError('No se encontró la variable objetivo y en el CSV de prueba.')
if raw.columns.duplicated().any():
    raise ValueError('Hay columnas duplicadas en el CSV de prueba.')

# Excluir y de las entradas.
y_test = raw['y'].to_numpy(dtype=np.int64)
X_test = raw.drop(columns=['y']).to_numpy(dtype=np.float32, copy=True)
if X_test.shape != (150, 193):
    raise ValueError(f'Esperados 150 registros × 193 predictores, recibidos {X_test.shape}.')
if not np.isfinite(X_test).all():
    raise ValueError('Existen valores no finitos en los predictores.')
if not set(np.unique(y_test)).issubset({0, 1}):
    raise ValueError('La etiqueta debe estar codificada como 0 y 1.')
if int(modelo.input_shape[-1]) != X_test.shape[1]:
    raise ValueError(f'El modelo espera {modelo.input_shape[-1]} predictores, pero TEST tiene {X_test.shape[1]}.')

# Probabilidad para cada fila, clase 1 = deserción; umbral de referencia = 0.50.
probabilidades = np.asarray(modelo.predict(X_test, verbose=0, batch_size=150)).reshape(-1)
if len(probabilidades) != len(y_test) or not np.isfinite(probabilidades).all():
    raise ValueError('La predicción produjo una salida de tamaño incorrecto o no finita.')
predicciones = (probabilidades >= 0.5).astype(int)
matriz = confusion_matrix(y_test, predicciones, labels=[0, 1]).tolist()
metricas_archivo = {
    'matriz_confusion': matriz,
    'accuracy': float(accuracy_score(y_test, predicciones)),
    'precision_desercion': float(precision_score(y_test, predicciones, zero_division=0)),
    'recall_desercion': float(recall_score(y_test, predicciones, zero_division=0)),
    'f1_desercion': float(f1_score(y_test, predicciones, zero_division=0)),
    'roc_auc': float(roc_auc_score(y_test, probabilidades)),
    'average_precision': float(average_precision_score(y_test, probabilidades)),
}
print('Arquitectura: entrada', modelo.input_shape, '| salida', modelo.output_shape)
print('Forma de TEST:', X_test.shape, '| Etiqueta y excluida de predictores: SÍ')
print('Resultados del archivo .keras efectivamente cargado:')
print(json.dumps(metricas_archivo, ensure_ascii=False, indent=2))
print('\nNota: las métricas publicadas de V1 pueden corresponder a otra ejecución de entrenamiento.')

entorno = {
    'fecha_hora_ejecucion': datetime.now().astimezone().isoformat(),
    'modelo_archivo': MODEL_NAME,
    'modelo_sha256': sha256(MODEL_PATH),
    'test_archivo': TEST_NAME,
    'test_sha256': sha256(TEST_PATH),
    'python': sys.version.split()[0],
    'tensorflow': tf.__version__,
    'keras': getattr(tf.keras, '__version__', 'integrado en TensorFlow'),
    'numpy': np.__version__,
    'pandas': pd.__version__,
    'sistema': platform.platform(),
    'cpu': platform.processor() or 'No identificado',
    'nucleos_logicos': os.cpu_count(),
    'gpu': [str(g) for g in tf.config.list_physical_devices('GPU')],
    'carga_modelo_ms': float(tiempo_carga_ms),
    'tipo_ejecucion': 'V1 reentrenada y persistida en .keras',
}

## 3. Protocolo de medición de latencia

Se realizan **10 calentamientos y 40 mediciones** para cada escenario de inferencia. Se registran media, mediana, desviación estándar, mínimo, máximo y P95. Se prueban solicitudes individuales y lotes de 150 registros, separando la conversión a tensor. Finalmente se mide la lectura del CSV ya preprocesado más la inferencia, con **5 calentamientos y 20 repeticiones**.

El cronómetro abarca la obtención real de resultados mediante `.numpy()`, que sincroniza las operaciones pendientes del backend, incluida la GPU. No se incluye el tiempo de carga del modelo en estas mediciones: se informa como indicador independiente. Las cifras son específicas del entorno de Colab utilizado.

In [ ]:
WARMUP = 10
REPETICIONES = 40

def inferencia_desde_tensor(t):
    salida = modelo(t, training=False)
    return np.asarray(salida.numpy())  # Materializa/sincroniza antes de detener el cronómetro.

def medir(escenario, n, funcion, warmup=WARMUP, repeticiones=REPETICIONES):
    for _ in range(warmup):
        salida = funcion()
        assert salida.shape[0] == n and np.isfinite(salida).all()
    tiempos = []
    for i in range(repeticiones):
        t0 = time.perf_counter_ns()
        salida = funcion()
        t1 = time.perf_counter_ns()
        assert salida.shape[0] == n and np.isfinite(salida).all()
        tiempos.append((t1 - t0) / 1e6)
    tiempos = np.asarray(tiempos, dtype=float)
    resumen = {
        'escenario': escenario, 'n_registros': n, 'repeticiones': repeticiones,
        'calentamiento': warmup, 'media_ms': float(tiempos.mean()),
        'mediana_ms': float(np.median(tiempos)), 'p95_ms': float(np.percentile(tiempos, 95)),
        'desviacion_ms': float(tiempos.std(ddof=1)),
        'min_ms': float(tiempos.min()), 'max_ms': float(tiempos.max()),
        'media_ms_por_registro': float(tiempos.mean() / n),
    }
    detalle = [{'escenario': escenario, 'repeticion': i+1, 'tiempo_ms': float(t)}
               for i, t in enumerate(tiempos)]
    return resumen, detalle

resumen, detalle = [], []
for etiqueta, datos in [('Individual', X_test[:1].copy()), ('Lote 150', X_test.copy())]:
    tensor_preparado = tf.convert_to_tensor(datos, dtype=tf.float32)
    escenarios = [
        (f'Inferencia pura - {etiqueta}', lambda t=tensor_preparado: inferencia_desde_tensor(t)),
        (f'Conversión + inferencia - {etiqueta}', lambda x=datos: inferencia_desde_tensor(tf.convert_to_tensor(x, tf.float32))),
    ]
    for titulo, funcion in escenarios:
        r, d = medir(titulo, len(datos), funcion)
        resumen.append(r)
        detalle.extend(d)

def csv_listo_y_prediccion():
    df = pd.read_csv(TEST_PATH, encoding='utf-8-sig')
    x = df.drop(columns=['y']).to_numpy(dtype=np.float32, copy=True)
    return inferencia_desde_tensor(tf.convert_to_tensor(x, dtype=tf.float32))

r, d = medir('Lectura CSV listo + inferencia - Lote 150', 150,
             csv_listo_y_prediccion, warmup=5, repeticiones=20)
resumen.append(r)
detalle.extend(d)
resumen_df = pd.DataFrame(resumen)
detalle_df = pd.DataFrame(detalle)
display(resumen_df.round(4))

## 4. Evidencia y descarga

Se generan una tabla de resumen, un CSV con todas las repeticiones, un JSON con configuración, métricas predictivas y huellas del modelo/dataset, una gráfica y un ZIP. **Se recomienda guardar este notebook con las salidas visibles además del ZIP.**

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(11, 5))
posiciones = np.arange(len(resumen_df))
ax.bar(posiciones, resumen_df['mediana_ms'], label='Mediana')
ax.scatter(posiciones, resumen_df['p95_ms'], marker='D', label='P95')
ax.set_xticks(posiciones)
ax.set_xticklabels(resumen_df['escenario'], rotation=25, ha='right')
ax.set_ylabel('Milisegundos por solicitud')
ax.set_title('MLP V1 reentrenada — latencia local de inferencia')
ax.grid(axis='y', alpha=.2)
ax.legend()
fig.tight_layout()
fig_path = OUTPUT_DIR / 'latencia_mlp_v1_colab.png'
fig.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()

ruta_resumen = OUTPUT_DIR / 'latencia_resumen.csv'
ruta_repeticiones = OUTPUT_DIR / 'latencia_repeticiones.csv'
ruta_json = OUTPUT_DIR / 'auditoria_latencia_mlp_v1.json'
resumen_df.to_csv(ruta_resumen, index=False)
detalle_df.to_csv(ruta_repeticiones, index=False)
reporte = {
    'alcance': 'PoC experimental, predicción sobre datos previamente transformados',
    'evaluacion_modelo_actual': metricas_archivo,
    'nota_trazabilidad': 'Esta medición corresponde exclusivamente al modelo cargado; no se atribuyen métricas de otras ejecuciones.',
    'entorno': entorno,
    'parametros_medicion': {'warmup_inferencia': WARMUP, 'repeticiones_inferencia': REPETICIONES,
                            'warmup_csv': 5, 'repeticiones_csv': 20},
    'resumen_tiempos': resumen,
    'excluye': ['Transformación de datos crudos', 'Red/API', 'Interfaz de usuario', 'Carga del modelo (informada aparte)'],
}
ruta_json.write_text(json.dumps(reporte, ensure_ascii=False, indent=2), encoding='utf-8')

zip_path = BASE_DIR / 'Evidencia_Medicion_Tiempo_MLP_V1_Colab.zip'
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    for ruta in [fig_path, ruta_resumen, ruta_repeticiones, ruta_json]:
        z.write(ruta, arcname=ruta.name)
print('Evidencia generada:')
for ruta in [ruta_resumen, ruta_repeticiones, ruta_json, fig_path, zip_path]:
    print('-', ruta)
print('NOTA: estos tiempos corresponden al .keras subido (segunda ejecución V1), no a la primera ejecución.')

if IN_COLAB:
    from google.colab import files
    files.download(str(zip_path))

## 5. Cómo incorporar los resultados al TFG

**Capítulo III (metodología):** describir equipo/Colab, versiones, modelo V1 persistido, forma de la matriz de TEST, uso de 193 variables sin `y`, calentamientos, repeticiones, mediana y P95. Indicar explícitamente que la medición corresponde a la **nueva ejecución** del modelo V1, no a la ejecución original utilizada para la comparación inicial V1–V2.

**Capítulo IV (resultados):** reportar la latencia individual y por lote del `.keras` medido, el equipo utilizado y que el tiempo excluye el preprocesamiento de datos crudos. No presentar la lectura del CSV listo como una medición del proceso completo institucional.

**Anexo:** conservar el notebook ejecutado y el ZIP de evidencia. La huella SHA-256 vincula el modelo físico con las mediciones. Las métricas de esta ejecución deben presentarse separadas de las métricas V1 originalmente documentadas.

**Advertencia científica:** el TEST ya fue observado en experimentos anteriores. Las métricas son descriptivas de este conjunto y no constituyen una evaluación externa inédita.